# Object Detection Walkthrough

Phase 2 of the platform: four from-scratch detectors sharing one interface —
**RetinaNet** (anchor-based one-stage), **FCOS** (anchor-free), **Faster R-CNN**
(two-stage), and **DETR** (transformer set prediction) — all trained and evaluated
on the fully offline synthetic shapes dataset, which a CPU can learn in minutes.

Every detector follows the same contract:
```
train mode:  model(images, targets) -> {"loss", ...}      # loss dict
eval mode:   model(images) -> [{"boxes", "scores", "labels"}]
```

In [ ]:
from pathlib import Path

if Path.cwd().name == "notebooks":
    %cd ..

import torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

## 1. The synthetic shapes dataset

Procedurally generated, deterministic per index, no downloads: rectangles, circles,
and triangles with tight bounding boxes.

In [ ]:
from image_analytics.data.datasets import DATASETS

ds = DATASETS.build("synthetic_shapes", size=8, image_size=96)
CLASSES = ds.CLASSES
COLORS = {0: "tab:red", 1: "tab:cyan", 2: "tab:olive"}

def draw_boxes(ax, boxes, labels, scores=None, color_by_label=True):
    for i, (box, label) in enumerate(zip(boxes, labels)):
        x1, y1, x2, y2 = box.tolist()
        color = COLORS[int(label)] if color_by_label else "yellow"
        ax.add_patch(mpatches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1, fill=False, color=color, linewidth=2))
        text = CLASSES[int(label)]
        if scores is not None:
            text += f" {scores[i]:.0%}"
        ax.text(x1, y1 - 2, text, color=color, fontsize=8)

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, idx in zip(axes, range(4)):
    image, target = ds[idx]
    ax.imshow(image.permute(1, 2, 0))
    draw_boxes(ax, target["boxes"], target["labels"])
    ax.axis("off")
plt.tight_layout()

## 2. Train Faster R-CNN

The two-stage detector — strongest of the three on this task. Swap the config path
for `retinanet_shapes.yaml` or `fcos_shapes.yaml` to compare paradigms; the rest of
the notebook is identical because all detectors share the interface.

In [ ]:
from image_analytics.core.config import load_config
from image_analytics.detection.train import run

config = load_config(
    "configs/detection/faster_rcnn_shapes.yaml",
    overrides=[
        "experiment_name=frcnn_notebook",
        "training.epochs=6",          # ~6 min CPU; bump for higher mAP
    ],
)
metrics = run(config)
metrics

## 3. Visualize predictions from the best checkpoint

In [ ]:
from image_analytics.data.transforms.detection import build_detection_transforms
from image_analytics.detection.train import build_detection_model

model = build_detection_model(config.model).eval()
ckpt = Path(config.output_dir) / config.experiment_name / "checkpoints" / "best.pt"
state = torch.load(ckpt, map_location="cpu", weights_only=True)
model.load_state_dict(state["model"])

val_tf = build_detection_transforms(config.data.image_size, train=False)
val_ds = DATASETS.build(
    "synthetic_shapes", split="val", size=8,
    image_size=config.data.kwargs["image_size"], transform=val_tf,
)
raw_ds = DATASETS.build(
    "synthetic_shapes", split="val", size=8,
    image_size=config.data.kwargs["image_size"],
)

fig, axes = plt.subplots(2, 4, figsize=(14, 8))
with torch.no_grad():
    for col in range(4):
        image, target = val_ds[col]
        raw_image, _ = raw_ds[col]
        pred = model(image.unsqueeze(0))[0]
        keep = pred["scores"] > 0.5

        axes[0, col].imshow(raw_image.permute(1, 2, 0))
        draw_boxes(axes[0, col], target["boxes"], target["labels"])
        axes[0, col].set_title("ground truth", fontsize=9)
        axes[1, col].imshow(raw_image.permute(1, 2, 0))
        draw_boxes(axes[1, col], pred["boxes"][keep], pred["labels"][keep],
                   pred["scores"][keep])
        axes[1, col].set_title("prediction", fontsize=9)
        for row in range(2):
            axes[row, col].axis("off")
plt.tight_layout()

## 4. Inside the pipeline

The pieces composing each detector are reusable on their own:

In [ ]:
from image_analytics.detection.necks.fpn import FPN
from image_analytics.detection.anchors import AnchorGenerator, Matcher
from image_analytics.backbones import build_backbone

backbone = build_backbone(
    "resnet18", pretrained=False, features_only=True,
)
fpn = FPN(backbone.feature_channels, out_channels=64, extra_levels="p6p7")

x = torch.rand(1, 3, 96, 96)
pyramid = fpn(backbone(x))
print("FPN pyramid:", [tuple(p.shape) for p in pyramid])

gen = AnchorGenerator(sizes=((16,), (32,), (64,), (128,), (256,)))
anchors = gen([tuple(p.shape[-2:]) for p in pyramid], strides=[8, 16, 32, 64, 128])
print("anchors per level:", [len(a) for a in anchors])

## 5. Where each detector shines

4-epoch CPU runs on shapes, random-init backbone (your numbers will vary slightly):

| Detector | mAP50 | mAP75 | Notes |
|---|---|---|---|
| RetinaNet | 0.35 | — | dense anchors + focal loss |
| FCOS      | 0.44 | 0.28 | anchor-free, GIoU + centerness localize better |
| Faster R-CNN | 0.59 | 0.40 | two-stage refinement wins at equal budget |
| DETR | — | — | needs 40-60 epochs on shapes (set prediction converges slowly) |

For real-time production detection, the `yolo` wrapper exposes the ultralytics
family through the same prediction protocol:

```python
from image_analytics.detection.yolo import YOLOWrapper
model = YOLOWrapper("yolo11n.pt")          # pretrained, downloads weights
predictions = model(images)                  # same dict protocol
model.train_native("data.yaml", epochs=50)  # ultralytics' own loop
```

## Next

Phase 3 (segmentation) builds on this directly: Mask R-CNN extends the
Faster R-CNN you just trained with a mask branch — see IMPLEMENTATION_PLAN.md.